# 🚀 Huấn luyện Mô hình XGBoost — Phát hiện bất thường mạng
**Đề tài:** Nghiên cứu ứng dụng học máy trong giám sát và phát hiện bất thường trên hệ thống mạng doanh nghiệp  
**Dữ liệu:** `data/processed/unified.parquet` (62 đặc trưng, 4 lớp: BENIGN, BruteForce, DoS/DDoS, PortScan)  
**Quy tắc:** Group-Aware Split bằng `StratifiedGroupKFold` theo `group_id` (chống rò rỉ dữ liệu)

In [ ]:
import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

# Đảm bảo import được module từ ml_engine
if '..' not in sys.path:
    sys.path.insert(0, '..')

from ml_engine.data_loader import get_train_val_test_split, CLASS_NAMES
from ml_engine.eval_utils import evaluate_predictions, measure_inference_latency, format_markdown_report

## 1. Nạp dữ liệu và Phân chia Train / Val / Test (Group-Aware)

In [ ]:
# Đường dẫn dữ liệu (tự động nhận diện local hoặc Kaggle)
parquet_path = '../data/processed/unified.parquet'
if not os.path.exists(parquet_path):
    parquet_path = '/kaggle/input/khoaluan-nids/unified.parquet'

# Để sample_ratio=None nếu muốn chạy toàn bộ 3.3 triệu dòng
X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = get_train_val_test_split(
    parquet_path=parquet_path,
    sample_ratio=None,
    random_state=42
)
print(f"Train size: {X_train.shape[0]:,} | Val size: {X_val.shape[0]:,} | Test size: {X_test.shape[0]:,}")
print(f"Số đặc trưng: {len(feature_cols)}")

## 2. Tính toán Sample Weight & Khởi tạo XGBoost

In [ ]:
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    early_stopping_rounds=15,
    random_state=42,
    n_jobs=-1
)

## 3. Huấn luyện mô hình

In [ ]:
start_time = time.time()
model.fit(
    X_train, y_train,
    sample_weight=sample_weights_train,
    eval_set=[(X_val, y_val)],
    verbose=10
)
train_duration = time.time() - start_time
print(f"Huấn luyện xong trong {train_duration:.2f} giây!")

## 4. Đánh giá toàn diện trên tập Test độc lập

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

results = evaluate_predictions(
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    model_name="XGBoost(hist, depth=6)",
    execution_time_sec=train_duration
)
latency = measure_inference_latency(model, X_test)

report_md = format_markdown_report(results, latency)
print(report_md)

## 5. Trực quan hóa Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

importance = model.feature_importances_
top_idx = np.argsort(importance)[::-1][:20]

plt.figure(figsize=(10, 8))
sns.barplot(x=importance[top_idx], y=[feature_cols[i] for i in top_idx], palette='viridis')
plt.title('Top 20 Đặc trưng quan trọng nhất (XGBoost Gain)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 6. Đóng gói mô hình .joblib

In [ ]:
save_dir = '../ml_engine/saved_models'
os.makedirs(save_dir, exist_ok=True)
joblib.dump({
    'model': model,
    'feature_cols': feature_cols,
    'class_names': CLASS_NAMES,
    'metrics': results
}, os.path.join(save_dir, 'xgboost_model.joblib'))
print("Đã đóng gói mô hình thành công!")